Table of content:

* A) Set-up & Configuration
* B) System Definition
* C) Routing Components / Orchestrator
* D) Standard evaluation
* E) Additional experiments


A) Set-up & Configuration

A1) Version Header

ThesisMiniMock version = 2.2.0 (Architecture V2 – Split Orchestration)

Goal:
This notebook implements a simplified orchestration layer to evaluate agent routing
strategies for a multi-tool LLM system in an enterprise procurement context.
The target system is currently in a design phase, and multiple orchestration architectures are under evaluation.

Architectural options under consideration include:
- V1: Unified orchestration architecture where a single routing layer selects amoung all available tools
- V2: Split orchestration architecture with two specialized routing paths corresponding to operational procurement data and knowledge-oriented information sources

This notebook focuses on Architecture V2 and evaluates a split orchestration design with improved routing models.

Each routing path is connected to a distinct subset of tools reflecting real system design
decisions currently under evaluation by the company.

Routing strategies evaluated:
- Embedding-based routing
- LLM-based routing
- Hybrid routing (rules → embeddings → LLM fallback)

Governance includes a confidence threshold with Human-in-the-Loop (HITL) escalation
when confidence is below threshold. Each routing decision produces a structured
trace to support explainability and offline evaluation.

This notebook builds on the single-agent baseline (V1) and enables direct comparison
of orchestration behavior, routing confidence, and governance outcomes between
single-agent and multi-agent designs.

Run order: top to bottom


2. Imports

In [5]:
# =============================================
# A2) Imports (Version 0.0.1)
# =============================================

# Standard librairies
import re
import json
import time
import io
from dataclasses import dataclass, field
from typing import Any, Dict, List, Tuple

# Data / numerical
import numpy as np
import pandas as pd

# Embeddings / similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Hugging Face LLM
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

3. Configuration constants

In [6]:
# ==============================
# A3) Configuration constants (V2.2)
# ==============================

EXPERIMENT_NAME = "ThesisMiniMock"
ARCHITECTURE = "V2_SPLIT_ORCHESTRATION"
VERSION = "2.2.0"

# Governance
CONFIDENCE_THRESHOLD = 0.80
ACTION_AUTO_ROUTE = "Auto_Route"
ACTION_HITL = "HITL_Escalation"

USE_FALLBACK = False
FALLBACK_THRESHOLD = 0.55

# Routing Strategies
STRATEGIES = ["Embedding_Based", "LLM_Router", "Hybrid"]
DEFAULT_STRATEGY = "Hybrid"

# V2-specific: orchestration paths
PATH_OPERATIONAL = "OperationalProcurement"
PATH_KNOWLEDGE = "StrategicKnowledge"

# Routing models
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
LLM_MODEL_NAME = "facebook/bart-large-mnli"

# Evaluation / reporting
TOP_K_EMBEDDING_SCORES_TO_SHOW = 2
PRINT_REASONING = True
PRINT_DECISION_TRACE = True

EXPORT_PATH = f"{EXPERIMENT_NAME}_{ARCHITECTURE}_v{VERSION}_results.csv"


In [7]:
# =============================================
# A4) Load Datasets
# =============================================

df_baseline = pd.read_csv("baseline_dataset.csv")
df_noisy = pd.read_csv("noisy_dataset.csv")
df_paraphrased = pd.read_csv("paraphrased_dataset.csv")
df_translated = pd.read_csv("translated_dataset.csv")

B) System Definition

1. Data Models
2. Tools catalog & Pool defintion
3. Query Preprocessing
4. Candidate Tools Selection

In [8]:
# ============================================================
# B1) Data Models (Version 2.0.0 - Architecture V2)
# ============================================================


@dataclass
class ToolPool:
    """Specialized tool pool associated with a subset of tools."""
    name: str
    description: str
    pool_type: str                   # "operational" or "knowledge"
    tools: List[Dict[str, str]]
    #metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class SelectionResult:
    """Intermediate model selection output before confidence-based governance is applied."""
    selected_pool: str
    selected_tool: str
    confidence: float
    trace: Dict[str, Any] = field(default_factory=dict)

@dataclass
class FinalResult:
    """Final system output after governance and logging."""
    query_id: int
    query: str
    strategy: str
    decision: SelectionResult
    action: str
    threshold: float


In [9]:
# ============================================================
# B2) Tool Catalog and Pool Configuration (Version 2.0.0 - Architecture V2)
# Split orchestration: Operational + Knowledge
# ============================================================

# Tool descriptions
GENERIC_DESC = "Answers generic procurement questions without calling structured sources."
GRBW_DESC = "Operational purchase-order line data (transactional)."
OA_DESC = "Outline agreement master data (contract types, validity, vendor, purchasing structure)."
OA_FULFILLMENT_DESC = "Outline agreement fulfillment & consumption KPIs (released/remaining budget, consumption quantities)."
SHAREPOINT_TRAINING_DESC = "Training materials and learning content from SharePoint (topics, responsibilities, sessions)."
ICERTIS_DESC = "Signed contract documents and extracted legal/contract metadata."
EMAIL_SHAREPOINT_DATA_DESC = "Email + SharePoint attachments analysis (pricing, suppliers, incoterms, quantities, etc.)."

# Tool split
TOOLS_OPERATIONAL = [
    {"name": "grbw_data_tool", "description": GRBW_DESC},
    {"name": "oa_data_tool", "description": OA_DESC},
    {"name": "oa_data_fulfillment_tool", "description": OA_FULFILLMENT_DESC},
]

TOOLS_KNOWLEDGE = [
    {"name": "generic_tool", "description": GENERIC_DESC},
    {"name": "sharepoint_training_tool", "description": SHAREPOINT_TRAINING_DESC},
    {"name": "icertis_data_tool", "description": ICERTIS_DESC},
    {"name": "email_sharepoint_data_tool", "description": EMAIL_SHAREPOINT_DATA_DESC},
]

TOOL_POOLS: List[ToolPool] = [
    ToolPool(
        name="OperationalProcurementPool",
        description="Handles SAP transactional queries involving specific busines objects such as purchase orders, outline agreements, vendor and delivery status. Use this for queries askingfor specific data or records.",
        pool_type="operational",
        tools=TOOLS_OPERATIONAL
    ),
    ToolPool(
        name="StrategicKnowledgePool",
        description="Handles high-level conceptual or explanatory questions about policies, definitions, and general concepts.",
        pool_type="knowledge",
        tools=TOOLS_KNOWLEDGE
    )
]

In [10]:
# ============================
# B3)  Query Preprocessing (V1.2)
# ============================

def normalize_text(text: str) -> str:
    """Lowercase, trim, collapse whitespace."""
    return re.sub(r"\s+", " ", text.strip().lower())

def preprocess_query(query: str) -> Tuple[str, Dict[str, Any]]:
    """Main preprocessing entry point before tool selection."""
    return normalize_text(query)


In [11]:
# ============================
# B4) Candidate Pool Selection (V2.2)
# ============================

def get_candidate_pools_v2(pools: List[ToolPool]) -> List[ToolPool]:
    """ V2: both pools are available to the first stage selector."""
    return pools

def get_tools_from_pool(pool: ToolPool) -> List[Dict[str, Any]]:
    """Returns the tools contained in the selected pool."""
    return pool.tools

C) Routing Engine

1. Router 1 : Emberdding based
2. Router 2: LLM-router (mock)
3. Hybrid cascade
4. Orchestrator wrapper

In [12]:
# ============================
# C1) Embedding Router (V2.2)
# ============================

class EmbeddingRouter:
    """
    Scores candidate tools using embedding cosine similarity between:
      - query text
      - tool.description text
    Embedding generated via Sentence Transformer.
    """
    def __init__(self, model_name: str, top_k: int = 3, use_fallback: bool = USE_FALLBACK, fallback_threshold=FALLBACK_THRESHOLD):
        self.top_k = top_k
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.use_fallback = use_fallback
        self.fallback_threshold = fallback_threshold
        use_fallback: bool = USE_FALLBACK
        fallback_threshold: float = FALLBACK_THRESHOLD

    def route(self, query: str, items: List[Dict[str, Any]], pool_name: str = "") -> SelectionResult:
        if not items:
            return SelectionResult(
                selected_tool="",
                selected_pool=pool_name if pool_name else "",
                confidence=0.0,
                trace={"strategy": "embedding", "error": "no_candidates"}
            )

        # Pool selection stage
        if pool_name == "":
            texts = [item.get("text", "") for item in items]
            names = [item.get("name", "") for item in items]

           # Encode query and candidate tool
            query_vec = self.model.encode([query], convert_to_numpy=True)
            item_vecs = self.model.encode(texts, convert_to_numpy=True)

            # Cosine simiarity: one score per candidate
            sims = cosine_similarity(query_vec, item_vecs).flatten()
            best_idx = int(np.argmax(sims))
            best_name = names[best_idx]
            best_score = float(sims[best_idx])

            # Trace: top-K scores
            top_idx = np.argsort(sims)[::-1][: self.top_k]
            top_scores = [(names[i], float(sims[i])) for i in top_idx]

            return SelectionResult(
                selected_pool=best_name,
                selected_tool="",
                confidence=best_score,
                trace={
                    "strategy": "embedding",
                    "model": self.model_name,
                    "top_scores": top_scores,
                    "num_candidates": len(items),
                    "level": "pool",
                    "fallback_enabled": self.use_fallback,
                    "fallback_threshold": self.fallback_threshold
            }
        )

        #Tool selection stage
        if self.use_fallback:
          specific_items = [item for item in items if item.get("name") != "generic_tool"]
          generic_item = next((item for item in item.get("name") == "generic_tool"), None)
        else:
          specific_items = items
          generic_item = None

        texts = [item.get("text", "") for item in items]
        names = [item.get("name", "") for item in items]

        # Encode query and candidate tool
        query_vec = self.model.encode([query], convert_to_numpy=True)
        item_vecs = self.model.encode(texts, convert_to_numpy=True)

        # Cosine simiarity: one score per candidate
        sims = cosine_similarity(query_vec, item_vecs).flatten()
        best_idx = int(np.argmax(sims))
        best_name = names[best_idx]
        best_score = float(sims[best_idx])

        # Trace: top-K scores
        top_idx = np.argsort(sims)[::-1][: self.top_k]
        top_scores = [(names[i], float(sims[i])) for i in top_idx]

        return SelectionResult(
            selected_pool=pool_name,
            selected_tool=best_name,
            confidence=best_score,
            trace={
                "strategy": "embedding",
                "model": self.model_name,
                "top_scores": top_scores,
                "num_candidates": len(items),
                "level": "tool",
                "fallback_enabled": self.use_fallback,
                "fallback_threshold": self.fallback_threshold
                }
            )

In [13]:
# ==========================================
# C2) Zero-ShotLLM Router (v2.2)
# ==========================================

class LLMRouter:

  def __init__(self, model_name: str, use_fallback: bool = USE_FALLBACK, fallback_threshold=FALLBACK_THRESHOLD):
    self.model_name = model_name
    self.use_fallback = use_fallback
    self.fallback_threshold = fallback_threshold
    self.classifier = pipeline(
        "zero-shot-classification",
        model=model_name
        )

  def route(self, query: str, items: List[Dict[str, Any]], pool_name: str = "") -> SelectionResult:
        #Pool selection stage
        if not items:
            return SelectionResult(
                selected_tool="",
                selected_pool=pool_name if pool_name else "",
                confidence=0.0,
                trace={"strategy": "llm", "error": "no_candidates"}
            )

        #Extract labels from tool descirtption
        if pool_name == "":
            labels = [item.get("text", "") for item in items]

            #Classify query
            result = self.classifier(
                query,
                candidate_labels=labels
            )

            best_text = result["labels"][0]
            confidence = float(result["scores"][0])

            #Map desciption back to pool name
            best_name = ""
            for item in items:
                if item.get("text", "") == best_text:
                    best_name = item.get("name", "")
                    break

            return SelectionResult(
                selected_pool=best_name,
                selected_tool="",
                confidence=confidence,
                trace={
                    "strategy": "llm",
                    "model": self.model_name,
                    "candidate_count": len(items),
                    "selected_text": best_text,
                    "level": "pool",
                    "fallback_used": self.use_fallback
                }
            )
        if self.use_fallback:
        #Tool selection stage
            specific_items = [item for item in items if item.get("name") != "generic_tool"]
            generic_item = next((item for item in items if item.get("name") == "generic_tool"), None)
        else:
            specific_items = items
            generic_item = None

        #Extract labels from tool descirtption
        labels = [item.get("text", "") for item in specific_items]

        #Classify query
        result = self.classifier(
            query,
            candidate_labels=labels
        )

        best_text = result["labels"][0]
        confidence = float(result["scores"][0])

        #Map desciption back to ool name
        best_name = ""
        for item in specific_items:
            if item.get("text", "") == best_text:
                best_name = item.get("name", "")
                break

        # Optional fallback
        if self.use_fallback and confidence < self.fallback_threshold:
          if best_name == "":
              best_name = generic_item["name"] if generic_item else specific_items[0]["name"]
          elif confidence < self.fallback_threshold and generic_item is not None:
              best_name = generic_item["name"]

        return SelectionResult(
            selected_tool=best_name,
            selected_pool=pool_name,
            confidence=confidence,
            trace={
                "strategy": "llm",
                "model": self.model_name,
                "candidate_count": len(items),
                "selected_description": best_text,
                "level": "tool",
                "fallback_enabled": self.use_fallback
            }
        )

In [14]:
## ============================
## C3) Hybrid Router (V2.2)
## ============================

class HybridRouter:
    """
    Hybrid cascade:
      1) embedding router
      2) LLM fallback if embedding confidence is below treshold
    """

    def __init__(self, embedding_router: EmbeddingRouter, llm_router: LLMRouter, threshold: float):
        self.embedding_router = embedding_router
        self.llm_router = llm_router
        self.threshold = threshold

    def route(
        self,
        query: str,
        items: List[Dict[str, Any]],
        pool_name: str = ""
    ) -> SelectionResult:
    #Step 1: Embedding Router
      emb = self.embedding_router.route(query, items, pool_name)
      emb.trace["cascade_stage"] = "embedding"

      if emb.confidence >= self.threshold:
        emb.trace["final_choice"] = "embedding"
        return emb

    #Step 2: LLM fallback
      llm = self.llm_router.route(query, items, pool_name)
      if llm is None:
          return emb #fallback to embeddinginstead of return None
      llm.trace["cascade_stage"] = "llm-fallback"
      llm.trace["prev_embedding_conf"] = emb.confidence
      llm.trace["embedding_suggestion"] = emb.selected_tool if pool_name else emb.selected_pool
      llm.trace["treshold"] = self.threshold

      return llm


In [15]:
# ============================
# C4) Orchestrator Wrapper (Version 2.2)
# ============================

class MiniOrchestratorV2_2:
    def __init__(self, tool_pools: List[ToolPool], threshold: float = CONFIDENCE_THRESHOLD):
        self.tool_pools = tool_pools
        self.threshold = threshold
        self.logs: List[FinalResult] = []

        self.embedding_router = EmbeddingRouter(
            model_name=EMBEDDING_MODEL_NAME,
            top_k=TOP_K_EMBEDDING_SCORES_TO_SHOW,
            use_fallback=USE_FALLBACK,
            fallback_threshold=FALLBACK_THRESHOLD
        )
        self.llm_router = LLMRouter(
            model_name=LLM_MODEL_NAME,
            use_fallback=USE_FALLBACK,
            fallback_threshold=FALLBACK_THRESHOLD
        )
        self.hybrid_router = HybridRouter(
            self.embedding_router,
            self.llm_router,
            threshold=self.threshold
        )

    def _pool_items(self) -> List[Dict[str, str]]:
        return [
            {
                "name": p.name,
                "text": p.description
            }
            for p in self.tool_pools
        ]

    def _tool_items_for_pool(self, selected_pool: ToolPool) -> List[Dict[str, str]]:
        return [
            {
                "name": t.get("name", ""),
                "text": t.get("description", "")}
            for t in selected_pool.tools
        ]

    def confidence_gate(self, confidence: float) -> str:
        return ACTION_AUTO_ROUTE if confidence >= self.threshold else ACTION_HITL

    def route(self, query_id: int, query: str, strategy=DEFAULT_STRATEGY) -> Dict[str, Any]:
        norm_q = preprocess_query(query)

        # Step 1: pool selection
        pool_items = self._pool_items()

        if strategy == "Embedding_Based":
            pool_decision = self.embedding_router.route(norm_q, pool_items)
        elif strategy == "LLM_Router":
            pool_decision = self.llm_router.route(norm_q, pool_items)
        else:
            pool_decision = self.hybrid_router.route(norm_q, pool_items)

        selected_pool = next(
            (p for p in self.tool_pools if p.name == pool_decision.selected_pool),
            self.tool_pools[0]
        )

        # Step 2: tool selection
        tool_items = self._tool_items_for_pool(selected_pool)

        if not tool_items:
            return {
                "query_id": query_id,
                "query": query,
                "strategy": strategy,
                "selected_pool": selected_pool.name,
                "pool_confidence": pool_decision.confidence,
                "selected_tool": None,
                "tool_confidence": 0.0,
                "action": ACTION_HITL,
                "threshold": self.threshold,
                "trace": {
                    "pool": pool_decision.trace,
                    "tool": {"error": "no_tools_for_pool"}
                }
            }

        if strategy == "Embedding_Based":
            tool_decision = self.embedding_router.route(norm_q, tool_items, selected_pool.name)
        elif strategy == "LLM_Router":
            tool_decision = self.llm_router.route(norm_q, tool_items, selected_pool.name)
        else:
            tool_decision = self.hybrid_router.route(norm_q, tool_items, selected_pool.name)

        if tool_decision is None:
            return {
                "query_id": query_id,
                "query": query,
                "strategy": strategy,
                "selected_pool": selected_pool.name,
                "pool_confidence": pool_decision.confidence,
                "selected_tool": None,
                "tool_confidence": 0.0,
                "action": ACTION_HITL,
                "threshold": self.threshold,
                "trace": {"error": "tool_decision_none"}
            }

        action = self.confidence_gate(tool_decision.confidence)

        return {
            "query_id": query_id,
            "query": query,
            "strategy": strategy,
            "selected_pool": pool_decision.selected_pool,
            "pool_confidence": pool_decision.confidence,
            "selected_tool": tool_decision.selected_tool,
            "tool_confidence": tool_decision.confidence,
            "action": action,
            "threshold": self.threshold,
            "trace": {
                "pool": pool_decision.trace,
                "tool": tool_decision.trace
            }
        }

D) Standard evaluation

* 1. Orchestration instantiation
* 2. Output Normalizer
* 3. Standard Experiment Set-
* 4. Run Standard Experiment
* 5. Evaluation

In [16]:
# ============================
# D1) Orchestrator Instantiation (Version 2.2)
# ============================

orch_v2 = MiniOrchestratorV2_2(
    tool_pools=TOOL_POOLS,
    threshold=CONFIDENCE_THRESHOLD
)

print("Orch_v2 ready:", [p.name for p in TOOL_POOLS])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Orch_v2 ready: ['OperationalProcurementPool', 'StrategicKnowledgePool']


In [17]:
# ============================
# D2) Output Normalizer (Version 1)
# ============================

def to_evaluation_format(out):
        return {
            "query_id": out.get("query_id"),
            "query": out.get("query"),
            "strategy": out.get("strategy"),
            "selected_pool": out.get("selected_pool"),
            "pool_confidence": out.get("pool_confidence"),
            "selected_tool": out.get("selected_tool"),
            "final_confidence": out.get("tool_confidence"),
            "action": out.get("action"),
            "threshold": out.get("threshold"),
            "trace": out.get("trace"),
        }

E) Experiments & Testing


In [18]:
# ================================
# D3) Standard Experiment Setup
# ================================

# 1) Datasets
DATASETS = {
    "baseline": df_baseline.to_dict("records"),
    "noisy": df_noisy.to_dict("records"),
    "paraphrased": df_paraphrased.to_dict("records"),
    "translated": df_translated.to_dict("records"),
}

# 2) Strategies
STRATEGIES = [
    "Embedding_Based",
    "LLM_Router",
    "Hybrid"
]

# 3) Orchestrator
ORCH = orch_v2

# 4) Output settings
EXPORT_PATH = "experiment_results.csv"

# 5) Debug flags
PRINT_TRACE = False

# 6) Expected mapping
EXPECTED_MAP = {
    row["id"]: row["expected_tool"]
    for row in df_baseline.to_dict("records")
    }


In [19]:
# ================================
# D4) Run Standard Experiment
# ================================

all_rows = []

for dataset_name, dataset in DATASETS.items():
    print (f"Running dataset: {dataset_name}")

    for strategy in STRATEGIES:
        print (f"Running strategy: {strategy}")

        for q in dataset:
            out = ORCH.route(
                query_id=q["id"],
                query=q["query"],
                strategy=strategy
            )

            row = to_evaluation_format(out)
            row["dataset"] = dataset_name
            all_rows.append(row)
df_results_v2 = pd.DataFrame(all_rows)


print("Evaluation complete.")
print("Rows:", len(df_results_v2))
df_results_v2.head

#export
df_results_v2.to_csv("df_results_v2.csv", index=False)
print("Saved df_results_v2")

from google.colab import files
files.download("df_results_v2.csv")

Running dataset: baseline
Running strategy: Embedding_Based
Running strategy: LLM_Router


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Running strategy: Hybrid
Running dataset: noisy
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Running dataset: paraphrased
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Running dataset: translated
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Evaluation complete.
Rows: 1950
Saved df_results_v2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# ================================
# D5) Evaluation of the results
# ================================

# Architecture label
ARCHITECTURE = "split"

# 1) Map expected tool
df_results_v2 = df_results_v2.copy()
df_results_v2["architecture"] = ARCHITECTURE
df_results_v2["expected_tool"] = df_results_v2["query_id"].map(EXPECTED_MAP)

# 2) Compute correctnes
df_results_v2["is_correct"] = df_results_v2["selected_tool"] == df_results_v2["expected_tool"]

# 3) Summary metrics
summary_v2 = (
    df_results_v2
    .groupby(["architecture", "dataset", "strategy"])
    .agg(
        accuracy=("is_correct", "mean"),
        confidence=("final_confidence", "mean"),
        auto_rate=("action", lambda x: (x == "ACTION_AUTO_ROUTE").mean()),
    )
    .reset_index()
)

print("Evaluation complete.")
print("Rows:", len(df_results_v2))
summary_v2

Evaluation complete.
Rows: 1950


,architecture,dataset,strategy,accuracy,confidence,auto_rate
0,split,baseline,Embedding_Based,0.360,0.550362,0.0
1,split,baseline,Hybrid,0.280,0.458295,0.0
2,split,baseline,LLM_Router,0.280,0.458295,0.0
3,split,noisy,Embedding_Based,0.265,0.531226,0.0
4,split,noisy,Hybrid,0.215,0.506615,0.0
5,split,noisy,LLM_Router,0.215,0.506615,0.0
6,split,paraphrased,Embedding_Based,0.315,0.541331,0.0
7,split,paraphrased,Hybrid,0.225,0.465119,0.0
8,split,paraphrased,LLM_Router,0.225,0.465119,0.0
9,split,translated,Embedding_Based,0.205,0.514316,0.0


In [26]:
# ================================
# E1) Additonal experiments
# ================================

EXPERIMENT_CONFIGS = [
    {"name": "thr_0.8_no_fallback", "threshold": 0.8, "use_fallback": False},
    {"name": "thr_0.7_no_fallback", "threshold": 0.7, "use_fallback": False},
    {"name": "thr_0.6_no_fallback", "threshold": 0.6, "use_fallback": False},
    {"name": "thr_0.8_fallback", "threshold": 0.8, "use_fallback": True},
]

all_rows = []

for config in EXPERIMENT_CONFIGS:

    print(f"\n=== Running experiment with config: {config['name']} ====")

    #Rebuild routers with config
    embedding_router = EmbeddingRouter(
        model_name=EMBEDDING_MODEL_NAME,
        use_fallback=config["use_fallback"],
        fallback_threshold=FALLBACK_THRESHOLD
        )
    llm_router = LLMRouter(
        model_name=LLM_MODEL_NAME,
        use_fallback=config["use_fallback"],
        fallback_threshold=FALLBACK_THRESHOLD
        )
    hybrid_router = HybridRouter(
        embedding_router=embedding_router,
        llm_router=llm_router,
        threshold=config["threshold"]
        )

    #Rebuild orchestrator
    orch = MiniOrchestratorV2_2(
        tool_pools=TOOL_POOLS,
        threshold=config["threshold"],
    )

    # Run experiment
    for dataset_name, dataset in DATASETS.items():
        print(f"Running dataset: {dataset}")
        for strategy in STRATEGIES:
            print(f"Running strategy: {strategy}")

            for q in dataset:
                out = orch.route(
                    query_id=q["id"],
                    query=q["query"],
                    strategy=strategy
                )
                row = to_evaluation_format(out)

                # Add experiment meta data
                row["dataset"] = dataset_name
                row["experiment_name"] = config["name"]
                row["threshold"] = config["threshold"]
                row["fallback_enabledk"] = config["use_fallback"]

                all_rows.append(row)

df_experiments_v2 = pd.DataFrame(all_rows)

print("Done.")
print("Rows:", len(df_experiments_v2))
print(df_experiments_v2.head())

#export
df_experiments_v2.to_csv("df_experiments_v2.csv")
print("Saved df_experiments_v2")
from google.colab import files
files.download("df_experiments_v2.csv")



=== Running experiment with config: thr_0.8_no_fallback ====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Running dataset: [{'id': 1, 'query': 'What are the distinct outline agreement types?', 'expected_tool': 'oa_data_tool'}, {'id': 2, 'query': 'Who is the Vendor for outline agreement 456781234?', 'expected_tool': 'oa_data_tool'}, {'id': 3, 'query': 'What material did 0001234567 buy?', 'expected_tool': 'oa_data_tool'}, {'id': 4, 'query': 'What material is 10012345?', 'expected_tool': 'oa_data_tool'}, {'id': 5, 'query': 'Who bought 10054321?', 'expected_tool': 'oa_data_tool'}, {'id': 6, 'query': 'Please give me the company code related to 0000001234?', 'expected_tool': 'oa_data_tool'}, {'id': 7, 'query': "What is the total released value for all positions for vendor '0000009876'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {'id': 8, 'query': "What is the released value for all positions for vendor '0000005432'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {'id': 9, 'query': "What is the total quantity ordered for vendor '0000005432'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
# ========================================
# E2) Evaluation of the experiment results
# ========================================

# Create a copy:
summary_exp_v2 = df_experiments_v2.copy()

# Create unified confidence column
if "final_confidence" in summary_exp_v2.columns:
    pass
elif "tool_confidence" in summary_exp_v2.columns:
    summary_exp_v2["final_confidence"] = summary_exp_v2["tool_confidence"]
elif "confidence" in summary_exp_v1.columns:
    summary_exp_v2["final_confidence"] = summary_exp_v2["confidence"]
else:
    summary_exp_v2["final_confidence"] = np.nan

summary_exp_v2["final_confidence"] = pd.to_numeric(summary_exp_v2["final_confidence"], errors="coerce")


# Create corecctness column if not already present
summary_exp_v2["expected_tool"] = summary_exp_v2["query_id"].map(EXPECTED_MAP)

if "is_correct" not in summary_exp_v2.columns:
    summary_exp_v2["is_correct"] = (
        summary_exp_v2["selected_tool"] == summary_exp_v2["expected_tool"]
    ).astype(float)

#Make fallback label better
summary_exp_v2["fallback_status"] = summary_exp_v2["fallback_enabledk"].map({
    True: "With fallback",
    False: "No fallback"
})

# Build summary table
summary_exp_v2["architecture"] = "split"
summary_exp_v2["is_auto_route"] =(
    summary_exp_v2["action"].astype(str).str.lower().isin(["auto_route", "auto route"])
).astype(float)

summary_exp_v2 = (
    summary_exp_v2
    .groupby(["architecture",
              "dataset",
              "strategy",
              "threshold",
              "fallback_status"
    ],
             dropna=False
    )
    .agg(
        accuracy=("is_correct", "mean"),
        avg_confidence=("final_confidence", "mean"),
        auto_rate=("is_auto_route", "mean"),
        n_queries=("query_id", "count")
    )
    .reset_index()
    )

# Round for display
summary_exp_v2["accuracy"] = summary_exp_v2["accuracy"].round(3)
summary_exp_v2["avg_confidence"] = summary_exp_v2["avg_confidence"].round(3)
summary_exp_v2["auto_rate"] = summary_exp_v2["auto_rate"].round(3)

# Sort nicely
summary_exp_v2 = summary_exp_v2.sort_values(
    by=["architecture", "threshold", "fallback_status", "dataset", "strategy"]
).reset_index(drop=True)

print("Evaluation complete:")
print("Rows:", len(summary_exp_v2))
summary_exp_v2

Evaluation complete:
Rows: 48


,architecture,dataset,strategy,threshold,fallback_status,accuracy,avg_confidence,auto_rate,n_queries
0,split,baseline,Embedding_Based,0.6,No fallback,0.360,0.550,0.260,50
1,split,baseline,Hybrid,0.6,No fallback,0.160,0.478,0.280,50
2,split,baseline,LLM_Router,0.6,No fallback,0.280,0.458,0.100,50
3,split,noisy,Embedding_Based,0.6,No fallback,0.265,0.531,0.120,200
4,split,noisy,Hybrid,0.6,No fallback,0.185,0.516,0.265,200
5,split,noisy,LLM_Router,0.6,No fallback,0.215,0.507,0.180,200
6,split,paraphrased,Embedding_Based,0.6,No fallback,0.315,0.541,0.175,200
7,split,paraphrased,Hybrid,0.6,No fallback,0.210,0.476,0.210,200
8,split,paraphrased,LLM_Router,0.6,No fallback,0.225,0.465,0.150,200
9,split,translated,Embedding_Based,0.6,No fallback,0.205,0.514,0.025,200


In [29]:
files.download("df_results_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>